# Regrid population data to 0.1° x 0.1°

Population data used in this example are at a 1km resolution [Gao, 2020](https://doi.org/10.7910/DVN/TLJ99B) or 0.0083° x 0.0083°. Aggregation to each 0.1° × 0.1° grid cell is accomplished by summing the central 12 × 12 cells [GBD, 2021](https://doi.org/10.6069/vkdr-qy60).

The choice of which population dataset should be based on the underlying climate scenario being used. Here we use the Shared Socioeconomic Pathway 2 (SSP2) to match the SSP2-4.5 scenario.

In [ ]:
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [ ]:
def expand_grid(pop_orig):
    # === Define resolution and bounds ===
    lat_res = 1 / 120  # 0.008333... degrees
    lon_res = 1 / 120

    # Generate full latitude and longitude ranges
    lat_full = np.arange(-90 + lat_res / 2, 90, lat_res)
    lon_full = np.arange(-180 + lon_res / 2, 180, lon_res)

    # Confirm lengths match the original grid
    print(f"Longitude points for pop_orig: {len(pop_orig.lon)}, Longitude points: {len(lon_full)}")  # Should match lon: 43200

    # === Create new empty DataArray with NaNs ===
    pop_full = xr.DataArray(
        data=np.full((len(lat_full), len(lon_full)), np.nan),
        coords={"lat": lat_full, "lon": lon_full},
        dims=["lat", "lon"],
        name="population"
    )

    # === Add population data to empty DataArray ===
    # Find indices where original lat/lon match the full grid
    lat_idx = np.searchsorted(lat_full, pop_orig.lat.values)
    lon_idx = np.searchsorted(lon_full, pop_orig.lon.values)

    # Use indexing to insert the population data
    pop_full.values[np.ix_(lat_idx, lon_idx)] = pop_orig.values

    assert np.allclose(
        pop_full.sel(
            lat=pop_orig.lat, lon=pop_orig.lon, method="nearest", tolerance=1e-5
        ),
        pop_orig,
        equal_nan=True
    )

    return pop_full

In [ ]:
# === Path config ===
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")

# Regrid each population year [2000, 2010, 2020, ..., 2100]
for year in range(2000, 2101, 10):
    print(f"Processing {year}")
    if year == 2000:
        pop = xr.open_dataset(f"{POP_DIR}baseYr_total_{year}.nc4")["Band1"]
    else:
        pop = xr.open_dataset(f"{POP_DIR}ssp2_total_{year}.nc4")["Band1"]

    pop_full = expand_grid(pop)

    # From GBD21 “...summing the central 12 × 12 population cells.”
    pop_regrid = pop_full.coarsen(lat=12, lon=12).sum()

    description = ("Global map of population counts (total populations) at "
                   "0.1ºx0.1º resolution, aggregated from 1km resolution by "
                   "A.F. Wells 2025, consistent with the Shared Socioeconomic "
                   "Pathways (SSPs).")
    cite = ("Gao, J. (2020). Global 1-km Downscaled Population Grids, "
            "SSP-Consistent Projections and Base Year, v1.01 (2000 - 2100)"
            " https://doi.org/10.7910/DVN/TLJ99B, Harvard Dataverse, V1")

    pop_regrid.attrs["description"] = description
    pop_regrid.attrs["citation"] = cite
    pop_regrid.to_netcdf(f"{POP_DIR}ssp2_coarse_grid_{year}.nc")

print("Finished processing population regridding")

In [ ]:
# Concatenate regridding population data to one file
tot_pop = []
years = range(2000, 2101, 10)

for year in years:
    print(f"Processing {year}")
    pop = xr.open_dataarray(f"{POP_DIR}ssp2_coarse_grid_{year}.nc")
    tot_pop.append(pop)

population_time_series = xr.concat(tot_pop,
                                   xr.DataArray(years, dims="year", name="year"))

description = ("Global time-series map of population counts (total populations)"
               " at 0.1ºx0.1º resolution, aggregated from 1km resolution by "
               "A.F. Wells 2025, consistent with the Shared Socioeconomic Pathways "
               "(SSPs).")
cite = ("Gao, J. (2020). Global 1-km Downscaled Population Grids, "
        "SSP-Consistent Projections and Base Year, v1.01 (2000 - 2100)"
        " https://doi.org/10.7910/DVN/TLJ99B, Harvard Dataverse, V1")

pop_regrid.attrs["description"] = description
pop_regrid.attrs["citation"] = cite

population_time_series.to_netcdf(f"{POP_DIR}ssp2_coarse_grid_{years[0]}-{years[-1]}.nc")

print("All processing complete.")